# Morphology

##### Autors: Josep de Cid & Gonzalo Recio

In [1]:
import nltk

from nltk import pos_tag
from nltk.metrics import jaccard_distance
from nltk.stem import WordNetLemmatizer

from scipy.stats import pearsonr

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/josepdecidrodriguez/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/josepdecidrodriguez/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/josepdecidrodriguez/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

First we read the dataset into a list of pairs containing the sentences *[(A1, B1), ..., (An, Bn)]*

In [2]:
sentences = []

with open('trial/STS.input.txt', mode='r') as f:
    for idx, line in enumerate(f.readlines()):
        _, a, b = line.strip().split('\t')
        sentences.append([a, b])
        
        print(f'{idx + 1}a) {a}')
        print(f'{idx + 1}b) {b}\n')

1a) The bird is bathing in the sink.
1b) Birdie is washing itself in the water basin.

2a) In May 2010, the troops attempted to invade Kabul.
2b) The US army invaded Kabul on May 7th last year, 2010.

3a) John said he is considered a witness but not a suspect.
3b) "He is not a suspect anymore." John said.

4a) They flew out of the nest in groups.
4b) They flew into the nest together.

5a) The woman is playing the violin.
5b) The young lady enjoys listening to the guitar.

6a) John went horse back riding at dawn with a whole group of friends.
6b) Sunrise at dawn is a magnificent view to take in if you wake up early enough for it.



### S2 version - Using Words

In Session 2 (S2) we computed Jaccard distance using simply words.

As a recall, the Jaccard coefficient measures the similarity S between finite sample sets, and is defined as the size of the intersection divided by the size of the union of the sample sets. This function calculates the distance D as 1 - S, allowing us to see that the distance value for the 3rd pair of sentences, for example, is smaller than the others as these sentences are quite similar.

However, this measure is not the best option to compute similarities, as e.g. two sentences with the same meaning but using synonyms will result into a high distance, while two sentences with different meaning can use the same words and give small distance.

In [3]:
def jaccard_distance_sentences(pair, lang='english'):
    return jaccard_distance(
        set(nltk.word_tokenize(pair[0], language=lang)),
        set(nltk.word_tokenize(pair[1], language=lang))
    )

distances_s2 = list(map(jaccard_distance_sentences, sentences))   

distances_s2

[0.6923076923076923,
 0.7368421052631579,
 0.5333333333333333,
 0.5454545454545454,
 0.7692307692307693,
 0.8620689655172413]

### S3 version - Using Lemmas

In Session 3 (S3) we are using lemmas for computing Jaccard distance.

In [4]:
def tag_pair_sentences(pair, lang):
    return (
        pos_tag(nltk.word_tokenize(pair[0], language=lang)),
        pos_tag(nltk.word_tokenize(pair[1], language=lang))
    )

We've also tried to apply lemmatization to other Parts of Speech such as adjectives but we didn't improve anything, at least for our data.

In [5]:
wnl = WordNetLemmatizer()

def lemmatize_token(token):
    if token[1][0] in ['N', 'V']:
        return wnl.lemmatize(token[0].lower(), pos=token[1][0].lower())
    return token[0]

def lemmatize_sentence(sentence):
    return list(map(lemmatize_token, sentence))

def lemmatize_pair_sentences(pair):
    return (
        lemmatize_sentence(pair[0]),
        lemmatize_sentence(pair[1])
    )

In [6]:
def jaccard_distance_sentences(pair, lang='english'):
    pair = tag_pair_sentences(pair, lang)
    pair = lemmatize_pair_sentences(pair)
    return jaccard_distance(set(pair[0]), set(pair[1]))

distances_s3 = list(map(jaccard_distance_sentences, sentences))    
distances_s3

[0.6923076923076923,
 0.6666666666666666,
 0.5333333333333333,
 0.5454545454545454,
 0.7692307692307693,
 0.8620689655172413]

## Comparison and analysis

Now that we have computed the Jaccard distances using Words and using Lemmas, we are going to calculate Pearson correlation between these distances and the *Golden Standard* scores to see which method (words or lemmas) computes better similarity.  **Pearson correlation** coefficient is a value between -1 to +1, where 1 is total positive linear correlation, 0 is no linear correlation, and −1 is total negative linear correlation. Therefore, we are interested in seeing which method (words or lemmas) obtains higher correlation.

We also plot the obtained distances in order to compare them.

In [7]:
refs = [0, 1, 2, 3, 4, 5]

print(' '*12, '|      ', (' '*7).join(['1 |', '2 |', '3 |', '4 |', '5 |', '6 |']))
print('-'*74)
print('S2 distances |', ' | '.join(list(map(lambda x: f'{x:.5}', distances_s2))), '|')
print('S3 distances |', ' | '.join(list(map(lambda x: f'{x:.5}', distances_s3))), '|')

print()

print(f'S2 correlation: {pearsonr(refs, distances_s2)[0]:.15}')
print(f'S3 correlation: {pearsonr(refs, distances_s3)[0]:.15}')

             |       1 |       2 |       3 |       4 |       5 |       6 |
--------------------------------------------------------------------------
S2 distances | 0.69231 | 0.73684 | 0.53333 | 0.54545 | 0.76923 | 0.86207 |
S3 distances | 0.69231 | 0.66667 | 0.53333 | 0.54545 | 0.76923 | 0.86207 |

S2 correlation: 0.396238977611923
S3 correlation: 0.490670810375692


Comparing the Jaccard distances, the only change we noticed is in the second pair of sentences, where using lemmas reduces the distance from 0.73684 to 0.66667. This is because of the verb *invade*, which appears as *invaded* in the second sentence, which is lemmatized into the cannonical form of the verb, thus we obtain a smaller Jaccard distance. However, any of the other sentences don't offer any option to decrease this distance via lemmatization.

Due to the decrease of the distance in one pair of sentences, comparing with the previous lab session, we got around a higher correlation coefficient with the *Golden Standard* of ~0.5 ( ~0.1 higher than the previous one), which means that there is some more linear correlation between the *Golden Standard* and the Jaccard distance using lemmatization. 

So we can conclude that using Lemmas allows us to detect similar words with inflectional variations better than using only Words and thus improving similarity computation.

### Questions

1. Which is better: words or lemmas?

In general terms, the usage of lemmas improves the performance, because using only words, we may lose information about inflectional variations of the same word that we can't use to measure similarity or distance.

2. Do you think that could perform better for any pair of texts?

Yes, but as we commented in the previous session, the usage of synonyms does not improve the results, as these are considered as totally different words with different cannonical forms, without taking into account the semantical similarity, so lemmatization doesn't improve in this cases.